# Delta Demo — Episode 14: VACUUM
### "The Mistake Is Still in History Forever... But Is the DATA Still Recoverable Forever?"

---
**Prerequisites:** None

**Runtime:** Databricks Free Edition

**Run Mode:** Run All

**Safe to rerun:** Yes

**Creates its own demo table:** `employees_ep14`

**Deletes only its own demo data:** Yes

---

**This notebook is self-contained.** It creates and uses its own Delta table (`employees_ep14`) in a dedicated path — nothing outside this path is ever touched, and no other episode needs to be run first.

**Learning Outcome:** By the end of this episode, viewers should be able to explain the difference between a file being logically removed from a Delta table's snapshot and being physically deleted from disk, and understand exactly what VACUUM does — and what it permanently costs you.

**Core Question:** We've spent three episodes proving that Delta keeps old, tombstoned files around after every UPDATE, DELETE, and CTAS. How long does that actually last — and what happens to Time Travel once those files are gone for good?

### Today's Journey
✔ Build a baseline table, then make a few changes that leave orphaned files behind

↓

✔ Confirm real orphaned garbage exists on disk right now

↓

✔ Run VACUUM DRY RUN — see what WOULD be deleted, nothing happens yet

↓

✔ Understand why the default 7-day retention window exists

↓

✔ Override it for this demo, and run a REAL VACUUM

↓

✔ Confirm the orphaned files are now permanently gone

↓

✔ Try to Time Travel to a version that needed those files — and watch it fail

# =====================================================
# STEP 0 — Setup (Self-Contained Reset)
# =====================================================

In [0]:
%sh
rm -rf /Volumes/workspace/delta_demo/demo_files/employees_ep14

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.delta_demo;
CREATE VOLUME IF NOT EXISTS workspace.delta_demo.demo_files;

# =====================================================
# STEP 1 — Create Baseline (5 Records)
# =====================================================

In [0]:
%python
from pyspark.sql import functions as F
import glob, os

table_path = "/Volumes/workspace/delta_demo/demo_files/employees_ep14"

baseline = spark.createDataFrame(
    [
        (1, 'Ravi', 25000),
        (2, 'Sridevi', 23000),
        (3, 'Uma', 35000),
        (4, 'Srik', 32000),
        (5, 'Kanth', 28000),
    ],
    "eno INT, ename STRING, sal INT"
).withColumn("sal", F.col("sal").cast("DECIMAL(10,2)"))

baseline.write.format("delta").mode("overwrite").save(table_path)

# =====================================================
# STEP 2 — Create Real Orphaned Files (A Few Ordinary Updates)
# =====================================================
Nothing dramatic here — just normal business updates. But per everything we've established across this series, each UPDATE leaves the OLD file version sitting on disk, tombstoned but not deleted.

###display current data from the table

In [0]:
%sql
select * from delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep14` order by eno;

eno,ename,sal
1,Ravi,25000.00
2,Sridevi,23000.00
3,Uma,35000.00
4,Srik,32000.00
5,Kanth,28000.00


In [0]:
%sql
describe history delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep14`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-08-02T01:34:41.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(4165969400519086),ce81febd-cc5c-4dc9-9328-6e1e20f14671,0802-001409-h47590in-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1310)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
%sql
UPDATE delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep14` SET sal = 27000 WHERE eno = 1;

num_affected_rows
1


In [0]:
%sql
UPDATE delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep14` SET sal = 30000 WHERE eno = 3;

num_affected_rows
1


In [0]:
%sql
DELETE FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep14` WHERE eno = 5;

num_affected_rows
1


In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep14`;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-08-02T01:37:05.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4165969400519086),efd7c694-06fa-4539-a917-fd0a952701c9,0802-001409-h47590in-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1312, p25FileSize -> 1290, numDeletionVectorsRemoved -> 1, minFileSize -> 1290, numAddedFiles -> 1, maxFileSize -> 1290, p75FileSize -> 1290, p50FileSize -> 1290, numAddedBytes -> 1290)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
5,2026-08-02T01:37:04.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,DELETE,"Map(predicate -> [""(eno#19593 = 5)""])",null,List(4165969400519086),efd7c694-06fa-4539-a917-fd0a952701c9,0802-001409-h47590in-v2n,4,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1314, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 929, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 384)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-08-02T01:37:00.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4165969400519086),0575a75b-9167-4560-a541-4d2673520eea,0802-001409-h47590in-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2530, p25FileSize -> 1312, numDeletionVectorsRemoved -> 1, minFileSize -> 1312, numAddedFiles -> 1, maxFileSize -> 1312, p75FileSize -> 1312, p50FileSize -> 1312, numAddedBytes -> 1312)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-02T01:36:59.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,UPDATE,"Map(predicate -> [""(eno#19263 = 3)""])",null,List(4165969400519086),0575a75b-9167-4560-a541-4d2673520eea,0802-001409-h47590in-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1998, numDeletionVectorsUpdated -> 0, scanTimeMs -> 889, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1220, rewriteTimeMs -> 1108)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-02T01:36:11.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4165969400519086),d7762be9-75a4-400e-9c27-721c9cf4fa1a,0802-001409-h47590in-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2536, p25FileSize -> 1310, numDeletionVectorsRemoved -> 1, minFileSize -> 1310, numAddedFiles -> 1, maxFileSize -> 1310, p75FileSize -> 1310, p50FileSize -> 1310, numAddedBytes -> 1310)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-02T01:36:09.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,UPDATE,"Map(predicate -> [""(eno#18918 = 1)""])",null,List(4165969400519086),d7762be9-75a4-400e-9c27-721c9cf4fa1a,0802-001409-h47590in-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3138, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1861, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1226, rewriteTimeMs -> 1258)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-08-02T01:34:41.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(4165969400519086),ce81febd-cc5c-4dc9-9328-6e1e20f14671,0802-001409-h47590in-v2n,null,WriteSeria

# =====================================================
# STEP 3 — Confirm Real Orphaned Garbage Exists Right Now
# =====================================================
Same active-vs-physical check from Episode 11's closing reveal — but this time it's the whole point of the episode, not a teaser.

In [0]:
%python
history_df = spark.sql(f"DESCRIBE HISTORY delta.`{table_path}`")
latest = history_df.orderBy(history_df.version.desc()).first()
active_files = len(spark.read.format("delta").load(table_path).inputFiles())
total_physical_files = len(glob.glob(f"{table_path}/*.parquet"))

print(f"Active files (what a query reads right now): {active_files}")
print(f"Physical files sitting on disk right now: {total_physical_files}")
print(f"Orphaned garbage: {total_physical_files - active_files} files")

if total_physical_files > active_files:
    print("\n✅ VERIFIED: real orphaned garbage exists — VACUUM has something")
    print("   genuine to clean up in this demo, not a manufactured example.")
else:
    print("\n❌ No orphaned garbage found — investigate before continuing.")

Active files (what a query reads right now): 1
Physical files sitting on disk right now: 6
Orphaned garbage: 5 files

✅ VERIFIED: real orphaned garbage exists — VACUUM has something
   genuine to clean up in this demo, not a manufactured example.


### Capture the Version We'll Try to Time Travel to Later
Pick an early version — one that needed a file that's about to become eligible for deletion.

In [0]:
%python
pre_vacuum_target_version = 0  # the very first commit
time_travel_check = spark.read.format("delta") \
    .option("versionAsOf", pre_vacuum_target_version).load(table_path)
print(f"Time travel to version {pre_vacuum_target_version} works right now:")
time_travel_check.orderBy("eno").show()

Time travel to version 0 works right now:
+---+-------+--------+
|eno|  ename|     sal|
+---+-------+--------+
|  1|   Ravi|25000.00|
|  2|Sridevi|23000.00|
|  3|    Uma|35000.00|
|  4|   Srik|32000.00|
|  5|  Kanth|28000.00|
+---+-------+--------+



# =====================================================
# STEP 4 — VACUUM DRY RUN: See What WOULD Be Deleted
# =====================================================
This lists candidate files. Nothing is actually deleted by a DRY RUN.

In [0]:
%sql
VACUUM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep14` DRY RUN;

path


**Likely result: an empty or near-empty list.** By default, VACUUM only considers files older than `delta.deletedFileRetentionDuration` — **7 days**. Everything we just created is only minutes old, so none of it qualifies for deletion yet, even though it's genuinely orphaned.

# =====================================================
# STEP 5 — Why 7 Days? And How to Override It (Demo Only)
# =====================================================
The default retention window exists to protect **concurrent readers/writers** — a long-running query that started reading an old snapshot needs those files to still exist until it finishes. Lowering this in production risks corrupting an in-flight read.

**For this demo only**, we override both the retention duration AND the safety check that normally blocks unsafely-low retention values. This is explicitly a demo-only technique — never do this against a live production table without understanding exactly who else might be reading it.

In [0]:
%sql
ALTER TABLE delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep14`
SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours');

In [0]:
try:
    value = spark.conf.get("spark.databricks.clusterUsageTags.clusterAllTags")
except Exception:
    value = "Configuration not available"

print(value)

print("Spark Version:", spark.version)
print("Running on:", spark.version)
print("Using Databricks Serverless Compute")
print("Serverless doesn't expose all cluster-level Spark configurations.")
print("We'll use RETAIN 0 HOURS directly in the VACUUM command.")

Configuration not available
Spark Version: 4.1.0
Running on: 4.1.0
Using Databricks Serverless Compute
Serverless doesn't expose all cluster-level Spark configurations.
We'll use RETAIN 0 HOURS directly in the VACUUM command.


In [0]:
%python
# This safety check exists specifically to stop you from doing what we're
# about to do accidentally. On Serverless compute, we'll bypass this by using
# the RETAIN 0 HOURS clause directly in the VACUUM command (Cell 27) —
# only ever appropriate in an isolated, non-production demo like this one.
print("✅ Skipping config set — will use RETAIN 0 HOURS in VACUUM command instead")

✅ Skipping config set — will use RETAIN 0 HOURS in VACUUM command instead


# =====================================================
# STEP 6 — Run the REAL VACUUM
# =====================================================

In [0]:
%sql
VACUUM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep14` RETAIN 0 HOURS;

path
dbfs:/Volumes/workspace/delta_demo/demo_files/employees_ep14


### VERIFY — Orphaned Files Are Now Physically Gone

In [0]:
%python
active_files_after = len(spark.read.format("delta").load(table_path).inputFiles())
total_physical_files_after = len(glob.glob(f"{table_path}/*.parquet"))

print(f"Active files: {active_files_after}")
print(f"Physical files on disk now: {total_physical_files_after}")

if total_physical_files_after == active_files_after:
    print("\n✅ VERIFIED: physical file count now matches active file count —")
    print("   the orphaned garbage is genuinely gone, not just hidden.")
else:
    print(f"\n⚠️ Still {total_physical_files_after - active_files_after} orphaned file(s) remaining.")

Active files: 1
Physical files on disk now: 1

✅ VERIFIED: physical file count now matches active file count —
   the orphaned garbage is genuinely gone, not just hidden.


# =====================================================
# STEP 7 — Try Time Travel Again — Does It Still Work?
# =====================================================
We proved this worked in Step 3. Let's try the EXACT same command now.

In [0]:
%python
try:
    result = spark.read.format("delta") \
        .option("versionAsOf", pre_vacuum_target_version).load(table_path)
    result.orderBy("eno").show()
    print(f"✅ Time travel to version {pre_vacuum_target_version} still works.")
except Exception as e:
    print(f"❌ Time travel to version {pre_vacuum_target_version} FAILED.")
    print(f"\nReal error:\n{str(e)[:500]}")

❌ Time travel to version 0 FAILED.

Real error:
[DELTA_UNSUPPORTED_TIME_TRAVEL_BEYOND_DELETED_FILE_RETENTION_DURATION] Cannot time travel beyond delta.deletedFileRetentionDuration (0 HOURS) set on the table.

JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaAnalysisException
	at com.databricks.sql.transaction.tahoe.DeltaErrorsBase.timeTravelBeyondDeletedFileRetentionDurationException(DeltaErrors.scala:1855)
	at com.databricks.sql.transaction.tahoe.DeltaErrorsBase.timeTravelBeyondDeletedFileRetentionDurationException$(DeltaErrors.scal


**Whichever way this actually goes, paste it back before it goes on a slide** — same discipline as every other episode. Whether version 0 specifically survived depends on whether the exact files it needed were among the ones just vacuumed. Either outcome is a valid, real teaching moment: if it fails, that's VACUUM's real cost, stated plainly, not softened. If it still works, that's proof the file it needed wasn't among the ones removed — worth explaining precisely which version DID break, by checking a few more with the same technique.

In [0]:
%sql
describe history delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep14/`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-08-02T01:39:53.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.deletedFileRetentionDuration"":""interval 0 hours""})",null,List(4165969400519086),a3a4710b-02e5-494c-96d9-084fc012b00b,0802-001409-h47590in-v2n,6,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
6,2026-08-02T01:37:05.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4165969400519086),efd7c694-06fa-4539-a917-fd0a952701c9,0802-001409-h47590in-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1312, p25FileSize -> 1290, numDeletionVectorsRemoved -> 1, minFileSize -> 1290, numAddedFiles -> 1, maxFileSize -> 1290, p75FileSize -> 1290, p50FileSize -> 1290, numAddedBytes -> 1290)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
5,2026-08-02T01:37:04.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,DELETE,"Map(predicate -> [""(eno#19593 = 5)""])",null,List(4165969400519086),efd7c694-06fa-4539-a917-fd0a952701c9,0802-001409-h47590in-v2n,4,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1314, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 929, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 384)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-08-02T01:37:00.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4165969400519086),0575a75b-9167-4560-a541-4d2673520eea,0802-001409-h47590in-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2530, p25FileSize -> 1312, numDeletionVectorsRemoved -> 1, minFileSize -> 1312, numAddedFiles -> 1, maxFileSize -> 1312, p75FileSize -> 1312, p50FileSize -> 1312, numAddedBytes -> 1312)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-02T01:36:59.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,UPDATE,"Map(predicate -> [""(eno#19263 = 3)""])",null,List(4165969400519086),0575a75b-9167-4560-a541-4d2673520eea,0802-001409-h47590in-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1998, numDeletionVectorsUpdated -> 0, scanTimeMs -> 889, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1220, rewriteTimeMs -> 1108)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-02T01:36:11.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4165969400519086),d7762be9-75a4-400e-9c27-721c9cf4fa1a,0802-001409-h47590in-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2536, p25FileSize -> 1310, numDeletionVectorsRemoved -> 1, minFileSize -> 1310, numAddedFiles -> 1, maxFileSize -> 1310, p75FileSize -> 1310, p50FileSize -> 1310, numAddedBytes -> 1310)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-02T01:36:09.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,UPDATE,"Map(predicate -> [""(eno#18918 = 1)""])",null,List(4165969400519086),d7762be9-75a4-400e-9c27-721c9cf4fa1a,0802-001409-h47590in-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3138, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1861, numAddedFiles -> 1, numUpdatedRows -> 1, nu

In [0]:
%sql
select * from delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep14/` order by eno;

eno,ename,sal
1,Ravi,27000.00
2,Sridevi,23000.00
3,Uma,30000.00
4,Srik,32000.00


# =====================================================
# STEP 8 — Enterprise Reality
# =====================================================
> "VACUUM is not optional cleanup — left unchecked, tombstoned files accumulate forever and quietly inflate storage costs. But it's also irreversible: once a file is physically deleted, no version of Delta Lake can bring it back. The 7-day default exists as a safety margin between 'this file is no longer needed' and 'nobody could possibly still be reading it.'"

**We just proved both sides of this tradeoff — cleanup happened, and it has a real, permanent cost to some slice of history.**

We've now covered how Delta records history, recovers from mistakes, and permanently lets go of the past. Next: what happens when two people try to change the same row at the same time?

That's exactly what we'll explore in the next episode: MERGE Internals.